# Chapter 2: Provisioning, PoC Mode and Monitoring

Welcome to Chapter 2 of the course 5 minutes to Federated Learning with NVIDIA FLARE!

In chapter 1, we learned how to develop a federated application using NVIDIA FLARE's core APIs, as well as how to configure it and run it in a simulated environment. In this chapter, we take it to the next level by introducing project provisioning, PoC mode and project monitoring in NVIDIA FLARE. We first introduce the concept of project provisioning and explain how FLARE generates a federated project for real-world scenario. Then we introduce the Proof-of-Concept (PoC) mode in NVIDIA FLARE, a suite of command-line tools that allow FL developers to run a provisioned project in a sandbox environment on a single PC. Next, walk you through NVIDIA FLARE's tools and APIs that allow you to monitor and track the progress of the federated project. Finally, we illustrate all these features with a hands-on example of CIFAR10 image classification using Pytorch.

After this chapter, you will:
- Have a basic understanding of project provision, PoC mode and project monitoring in NVIDIA FLARE
- Be able to generate a federated project that can be deployed as a real-world application 
- Be able to run a provisioned federated job in a sanbox environment 
- Be able to track and monitor a running federated project using FLARE's tools and APIs


# Provisioning in NVIDIA FLARE

In Chapter 1, we saw togther how to develop an example federated application using NVIDIA FLARE's APIs, and run the application in a simulated environment using FL Simulator. In that example, we defined 1 server and 2 clients as participants of the FL project. The generation and management of the participants are all managed by the FL Simulator tool, and we didn't really have the need to set them up. While this is all convenient for prototyping, for real-world federated applications, we need more fine-grained control in setting up different participants. In real-world FL deployment, different participants have their own clearly defined roles and policies that are managed by the participants themselves. Apart from server and clients, FL projects are usually assigned with administrators who manage and monitor project status. Proper user authentication and authorization mechanisms need to be set up to ensure a secure deployment environment for all participants to establish mutual-trusted communication channels. Also, due to the distributed nature of FL projects, all participants would need to be able to join the FL system from different locations through secure network connections. On top of all these, extra security and resource management features might also be required due to different participants requirements in terms of privacy and compute resources.

To have a managed way to properly set up an FL project and account for all the aspects mentioned above, NVIDIA FLARE introduces the concept of [**Provisioning**](https://nvflare.readthedocs.io/en/main/programming_guide/provisioning_system.html#provisioning-in-nvidia-flare). In FLARE, **provisioning is the process of settting up a federated project and participants for a scalable, extensible and secure real world deployment**. Provisioning in NVIDIA FLARE is used to:
- Establish the identities of the server, clients, and administrators in the federation
- Generate startup kit for each participant
- Set up secure communication channels between participants

<img src="../images/provision.png" alt="NVFLARE Provision" width=30% />

In FLARE, provisioning can be done using either the CLI tool `nvflare provision` or webUI-based [Dashboard](https://nvflare.readthedocs.io/en/main/user_guide/dashboard_ui.html). In this chapter, we mainly focus on the CLI tool. We will cover Dashboard in the next chapter. The provisioning process typically involves the following steps:
1. Configure the project using a configuration file
2. Use the FLARE provisioning tool to generate startup kits from project configuration file
3. Distribute the startup kits to participants

Let's walk through these steps together.

### 1. Configure the project

The input to the povisioning process is a project configuration file (for instance typically `project.yaml`). This configuration is used to set up the FL project, and generally includes definitions for:
- Project meta-data
- Participants of the project
- Builders to build project workspace

An example project configuration file is provided in [`files/project.yml`/](files/project.yml/). This file was generated using the `nvflare provision` CLI tool without any arguments. Noted that when running this CLI tool without arguments, you will be prompted to select whether the generated configuration file should include *high-availability* features. The example file was generated without *high-availability* features. More details on *high-availability* will be covered in the next chapter. You can also master templates for more complete project configurations [here](https://nvflare.readthedocs.io/en/main/programming_guide/provisioning_system.html#default-project-yml-file).

Let us look at the content of [`files/project.yaml`/](files/project.yml/) together (some of the details and comments are removed to faciliate explanation):
```yaml
api_version: 3
name: example_project
description: NVIDIA FLARE sample project yaml file

participants:
  - name: server1
    type: server
    org: nvidia
    fed_learn_port: 8002
    admin_port: 8003

  - name: site-1
    type: client
    org: nvidia

  - name: site-2
    type: client
    org: nvidia
    
  - name: admin@nvidia.com
    type: admin
    org: nvidia
    role: project_admin

builders:
  - path: nvflare.lighter.impl.workspace.WorkspaceBuilder
      ...
  - path: nvflare.lighter.impl.template.TemplateBuilder
      ...
  - path: nvflare.lighter.impl.static_file.StaticFileBuilder
      ...
  - path: nvflare.lighter.impl.cert.CertBuilder
      ...
  - path: nvflare.lighter.impl.signature.SignatureBuilder
```

The configuration file is organized into the following 3 sections:

**Project mete-data**
  - The `api_version`: for current release of NVIDIA FLARE, the `api_version` is set to 3
  - The `name` of the FL project, in this case, `example_project`
  - And a short `description` of the FL project

**Participants**

This section is a crucial part that defines all the [participants](https://nvflare.readthedocs.io/en/main/programming_guide/provisioning_system.html#participant) involved in the project. There are multiple types of participants defined in FLARE, most common types include `server`, `client` and `admin`. Some participants are referred to as [Sites](https://nvflare.readthedocs.io/en/main/user_guide/security/terminologies_and_roles.html#site), which represent computing system that runs FLARE applications, for instance, the `server` and `client`. While some participants are referred to as [Users](https://nvflare.readthedocs.io/en/main/user_guide/security/terminologies_and_roles.html#user), who are human participants, with different access priviledges to query, monitor or manage the project, for instance, the `admin`. Developers can aslo include other types for instance the `overseer` for high-availability mode, or even add custom types, but we will not go into details on this in this course. 

As we can see in the example `project.yml` file, the following participants are defined:
- `server`: here we defined 1 server. The name of the server should in general be a [fully qualified domain name](https://en.wikipedia.org/wiki/Fully_qualified_domain_name) (FQDN) to make sure that other participants can establish network connections to the server from any location. The server name can also be a system-wide known hostname.
- `client`s: here we defined 2 clients
- `admin`: here we defined 1 `project_admin`, who has the most elevated access priviledges in the whole project. There can only 1 project admin for each project.

**Builders**

In NVIDIA FLARE provisioning, [builders](https://nvflare.readthedocs.io/en/main/programming_guide/provisioning_system.html#builder) are a series of Python classes that work together to generate startup kits for all participants, based on the configurations defined in the project configuration file. Builds create various aspects of the startup kits for all participants, such as workspace structure, configuration files, and security credentials. Builders specified in a configuration file will be invoked in the same order as listed in the `builders` section. This example `project.yml` file shows the usage of common builders:
- `WorkspaceBuilder`: Creates the basic workspace structure
- `TemplateBuilder`: Processes template files
- `StaticFileBuilder`: Copies static files into the workspace
- `CertBuilder`: Generates certificates and keys for secure communication
- `SignatureBuilder`: Creates signatures for tamper-proofing

Developers can also create custom builders to construct more specific and customized provisioning output. We will not deep dive into how builders work and how they can be customized. For now, the most important thing to know about builders is that they execute in specific order to build the startup kits as results of a provisioning process.


### 2. Generate startup kits

Let's run the provisioning process with the example configuration file [`files/project.yml`/](files/project.yml/):

In [1]:
!rm -rf ./temp-workspace
!nvflare provision -p files/project.yml -w ./temp-workspace

Project yaml file: /flare/notebooks/files/project.yml.
Generated results can be found under /flare/notebooks/./temp-workspace/example_project/prod_00. 


Now let's check the structure of the output generated by provisioning:

In [2]:
!tree /tmp/temp-workspace -L 4

/tmp/temp-workspace
└── example_project
    ├── prod_00
    │   ├── admin@nvidia.com
    │   │   ├── local
    │   │   ├── startup
    │   │   └── transfer
    │   ├── server1
    │   │   ├── local
    │   │   ├── readme.txt
    │   │   ├── startup
    │   │   └── transfer
    │   ├── site-1
    │   │   ├── local
    │   │   ├── readme.txt
    │   │   ├── startup
    │   │   └── transfer
    │   └── site-2
    │       ├── local
    │       ├── readme.txt
    │       ├── startup
    │       └── transfer
    ├── resources
    │   ├── aws_template.yml
    │   ├── azure_template.yml
    │   └── master_template.yml
    └── state
        └── cert.json

21 directories, 7 files


We can see that provisioning generates a well-defined directory hierarchy.

At the top level, we have a root folder with the FL project's name, in this case, `example_project`. Under `example_project`, we can see multiple subfolders:
- `resources`: this directory may contain shared resources or additional files needed for the project.
- `state`: this is a directory to maintain state information about the provisioning process or current status of participants.
- `prod_NN`: this folder contains the **startup kits** generated from a successful provisioning command for all participants. The number (NN) increases with each successful provision run, indicating different provisioning sessions. In this example case, since we are running the provisioning command for the first time, the folder name is `prod_00`.

**A startup kit is a folder with a set of files, scripts and credentials generated by provisioning for each participant in the FL project**. A startup kit typically include:
- Authentication credentials
- Authorization policies
- Signatures for tamper-proof mechanisms
- Convenient shell scripts for launching participants

We can see that in this example, there are 4 startup kits generated, one for each participants, i.e., 1 server, 2 clients and 1 admin. Each folder is named after its corresponding participant's name as indicated in the configuration file. 

Let's look into the content of the generated startup kits for the server, clients and admin. Server and clients have similar startup kit content structure. Let's display the server startup kit files with the following command:


In [3]:
!tree ./temp-workspace/example_project/prod_00/server1/

./temp-workspace/example_project/prod_00/server1/
├── local
│   ├── authorization.json.default
│   ├── log.config.default
│   ├── privacy.json.sample
│   └── resources.json.default
├── readme.txt
├── startup
│   ├── fed_server.json
│   ├── rootCA.pem
│   ├── server.crt
│   ├── server.key
│   ├── signature.json
│   ├── start.sh
│   ├── stop_fl.sh
│   └── sub_start.sh
└── transfer

4 directories, 13 files


Different subfolders are organized as follows: 
- The `local` subfolder: this folder contains configuration files for site-specific local policies for authorization, logging, privacy and resource access. Each site can modify these configuration files to set up their local policies.
- The `startup` subfolder: this folder contains shell script for a participant to start / join (`start.sh`) or leave the FL project (`stop_fl.sh`). It also contains certificates, signature and key files, which are essential to maintain secure connections between different participants. Noted that the signatures and certificates files are integral to ensuring the security and authenticity of a participants. Any modification of these files after provisioning will prevent a participant to connect to the FL system. 
- The `transfer` subfolder is used to store artifacts such as custom application, scripts or files during project runtime.

Now let's look at the content of `admin`'s startup kit:

In [4]:
!tree /tmp/temp-workspace/example_project/prod_00/admin@nvidia.com/

/tmp/temp-workspace/example_project/prod_00/admin@nvidia.com/
├── local
├── startup
│   ├── client.crt
│   ├── client.key
│   ├── fed_admin.json
│   ├── fl_admin.sh
│   ├── readme.txt
│   └── rootCA.pem
└── transfer

4 directories, 6 files


The `admin`'s startup kit has similar structure. 
- The `local` subfolder is empty, since local policies are not needed for admin, a human participant who manages the project.
- In the `startup` subfolder, apart from certificates and keys, the `fl_admin.sh` shell script allows the `admin` user to login to [**FLARE Console**](https://nvflare.readthedocs.io/en/main/real_world_fl/operation.html#operating-nvflare) to perform management tasks. We will look at the **FLARE Console** later.
- The `transfer` folder in `admin`'s startup kit can be used later to link or hold any federated job / application developed using FLARE, so that later the job / application can be submitted to server for running. We will see how this works in detail later.

### 3. Distribute the startup kits to participants

Now with startup kits generated, the last step is to distribute these kits to the corresponding participants. You can use email, sftp etc. to do so, as long as you can ensure that it is secure. In general, Each site should have an organization admin to receive or download the startup kits. The organization admin can then install their own packages, start the services, map the data location, and instrument the authorization policies and organization level site privacy policies. We won't go into details of this during this course. 

**That's it! We have learned how to properly set up a federated project and generate startup kits for all participants using FLARE's provisioning process!**

# Proof-of-Concept Mode

Now let's put what we've learn about FLARE provisioning into action: let us run a federated application with properly defined server, clients and admin!

While it sounds exciting, this is however not a trivial task. We would have to set up distributed computing environments for the server, clients and admin. It would be nice if there is a way to alleviate this set-up. Introducing NVIDIA FLARE's Proof-of-Concept, or [PoC mode](https://nvflare.readthedocs.io/en/main/user_guide/poc_command.html). PoC mode allows users to test the features of FLARE provisioning and deployment on a single machine, without the overhead of a true distributed deployment and the need to establish secure communication between server and client systems.

Compared to the FL Simulator, where the job run is automated on a single system, PoC mode allows you to establish and connect distinct server, client and admin in different processes. PoC mode also opens possibility for admin to orchestrate and manage the project using the FLARE Console, making it a useful tool in preparation for a real-world distributed deployment.

Developers often start with PoC mode to test their applications locally before transitioning to a production environment where provisioning is essential. The transition involves moving from a simplified setup to a more secure configuration that includes mutual authentication and authorization policies.

To get started, let's look at the NVFlare CLI usage for PoC mode:

In [5]:
!nvflare poc -h

usage: nvflare poc [-h] [--prepare] [--start] [--stop] [--clean]
                   {prepare,prepare-jobs-dir,start,stop,clean} ...

options:
  -h, --help            show this help message and exit
  --prepare             deprecated, suggest use 'nvflare poc prepare'
  --start               deprecated, suggest use 'nvflare poc start'
  --stop                deprecated, suggest use 'nvflare poc stop'
  --clean               deprecated, suggest use 'nvflare poc clean'

poc:
  {prepare,prepare-jobs-dir,start,stop,clean}
                        poc subcommand
    prepare             prepare poc environment by provisioning local project
    prepare-jobs-dir    prepare jobs directory
    start               start services in poc mode
    stop                stop services in poc mode
    clean               clean up poc workspace


As we can see, the PoC mode is composed of multiple subcommands. Let's quickly walk through each of the subcommands.

#### `nvflare poc prepare`

This is the first command to run when launching a PoC mode project. This command generates a local project workspace with startup kits for all participants. Internally it runs the provision process either with a user-specified configuration file or with a default one.

Here is the help info for `nvflare poc prepare`:

In [6]:
!nvflare poc prepare -h

usage: nvflare poc prepare [-h] [-n [NUMBER_OF_CLIENTS]] [-c [CLIENTS ...]]
                           [-he] [-i [PROJECT_INPUT]] [-d [DOCKER_IMAGE]]
                           [-debug]

options:
  -h, --help            show this help message and exit
  -n [NUMBER_OF_CLIENTS], --number_of_clients [NUMBER_OF_CLIENTS]
                        number of sites or clients, default to 2
  -c [CLIENTS ...], --clients [CLIENTS ...]
                        Space separated client names. If specified,
                        number_of_clients argument will be ignored.
  -he, --he             enable homomorphic encryption.
  -i [PROJECT_INPUT], --project_input [PROJECT_INPUT]
                        project.yaml file path, If specified,
                        'number_of_clients','clients' and 'docker' specific
                        options will be ignored.
  -d [DOCKER_IMAGE], --docker_image [DOCKER_IMAGE]
                        generate docker.sh based on the docker_image, used in
            

By default, the command creates a workspace in `/tmp/nvflare/poc`. However, we can override the environment variable `NVFLARE_POC_WORKSPACE` to modify the workspace location. 

It is optional to specify a configuration file with the `-i` argument: if unspecified, a default template will be used to provision startup kits for 1 server, 2 clients and 1 admin.

#### `nvflare poc prepare-jobs-dir`

This command can be used to link any directory with federated applications / jobs developed using FLARE, so that they can be submitted by `admin`. As we've explained earlier, federated applications need to appear inside the `transfer` folder of the `admin`'s startup kit for the `admin` to be able to submit them to the server. This command essentially creates a symbolic link in `admin`'s `transfer` folder pointing to a directory of federated applications.

Here is the help info for `nvflare poc prepare-jobs-dir`:

In [10]:
!nvflare poc prepare-jobs-dir -h

usage: nvflare poc prepare-jobs-dir [-h] [-j [JOBS_DIR]] [-debug]

options:
  -h, --help            show this help message and exit
  -j [JOBS_DIR], --jobs_dir [JOBS_DIR]
                        jobs directory
  -debug, --debug       debug is on


#### `nvflare poc start` and `nvflare poc stop`

As the names indicate, these command starts / stops the FL system. You can either start / stop all participants, or a specific one, for instance, starting / stopping only the `server`.

Here is the help info for `nvflare poc start` and `nvflare poc stop`:

In [1]:
!nvflare poc start -h

usage: nvflare poc start [-h] [-p [SERVICE]] [-ex [EXCLUDE]] [-gpu [GPU ...]]
                         [-debug]

options:
  -h, --help            show this help message and exit
  -p [SERVICE], --service [SERVICE]
                        participant, Default to all participants
  -ex [EXCLUDE], --exclude [EXCLUDE]
                        exclude service directory during 'start', default to ,
                        i.e. nothing to exclude
  -gpu [GPU ...], --gpu [GPU ...]
                        gpu device ids will be used as CUDA_VISIBLE_DEVICES.
                        used for poc start command
  -debug, --debug       debug is on


In [2]:
!nvflare poc stop -h

usage: nvflare poc stop [-h] [-p [SERVICE]] [-ex [EXCLUDE]] [-debug]

options:
  -h, --help            show this help message and exit
  -p [SERVICE], --service [SERVICE]
                        participant, Default to all participants
  -ex [EXCLUDE], --exclude [EXCLUDE]
                        exclude service directory during 'stop', default to ,
                        i.e. nothing to exclude
  -debug, --debug       debug is on


By default, the PoC mode starts and stops all participants at once. You can use the `-p` or `-ex` flags to include or exclude certain participants.

#### `nvflare poc clean`

This command cleans up and deletes the POC workspaces.

# Job Submission and Project Monitoring

Now we've seen how to run an FL system with server, clients and admin in PoC mode, let's learn how to submit a federated job and monitor its running status in NVIDIA FLARE.

The first thing we need to do is to log into the FL system as the `admin` user. After all, job submission and monitoring are performed by users with certain access rights to the FL system. For instance, the project administrator. After logging into the FL system as `admin`, we will be able to manage the project, submit jobs and monitor the project runtime status. 

NVIDIA FLARE offers 2 ways to achieve the above:
1. Using `admin`'s startup shell script and [`FLARE Console`](https://nvflare.readthedocs.io/en/main/real_world_fl/operation.html#operating-nvflare)
2. Using [`FLARE API`](https://nvflare.readthedocs.io/en/main/real_world_fl/flare_api.html)

For now, let's focus on the first approach using `admin`'s startup shell script and `FLARE Console`. We will leave `FLARE API` as one of the exercises below for developers who want to dig deeper into the programmatic way of managing FL projects.


To connect to the FL system, you can either use the `nvflare poc start` subcommand, and specify only starting the `admin` user:
```shell
nvflare poc start -p admin@nvidia.com
```
Or you can directly invoke the shell script `fl_admin.sh` located in the `admin` user's startup kit:
```shell
./temp-workspace/example_project/prod_00/admin@nvidia.com/startup/fl_admin.sh
```

After connecting as the `admin` user, you will enter the [FLARE Console](https://nvflare.readthedocs.io/en/main/real_world_fl/operation.html#admin-command-prompt).

<img src="../images/flare-console.png" alt="FLARE Console" width=60% />

FLARE Console is a consle-like management interface for orchestrating federated projects. FLARE Console offers various commands allowing users to manage and control the state of the FL system, including starting and stopping servers and clients, deploying applications, and monitoring their status. With FLARE Console, it is easy to:
- Orchestrate the entire federated learning study, allowing users to initiate and manage various components of the system.
- Deploy applications to clients and servers from the console, facilitating the execution of collaborative tasks.
- Monitor project status: the console provides real-time updates on the status of servers and clients, enabling administrators to track progress and troubleshoot issues.

FLARE Console typically runs as a standalone process on a researcher’s workstation or laptop. It interacts only with the server, not directly with FL clients. Administrators can issue commands anywhere through the console to manage experiments effectively.

FLARE Console offers a comprehensive list of commands to manage and monitor jobs and the overall FL system. [Here](https://nvflare.readthedocs.io/en/main/real_world_fl/operation.html#admin-command-prompt) is the list of commands available in FLARE Console. Commonly used ones include:
- `?`: use the question mark to list all available commands in FLARE Console
- `check_status`: check the status of the server or the clients
- `info`: list directories for job / application upload & download
- `submit_job`: submit a federated job / application that is visible inside the `admin`'s transfer folder
- `list_jobs`: list jobs on the server 
- `download_job`: download results of a job using the job's ID
- `shutdown`: shutdown server or specific clients
- `bye`: log out of the FLARE Console


In [ ]:
!nvflare 

# Example: Federated CIFAR10 Image Classification Using Pytorch

Let's walk through together how to launch a federated application in PoC mode step-by-step. 

In [ ]:
!nvflare poc prepare -h

By default, the command creates a workspace in `/tmp/nvflare/poc`. However, we can override the environment variable `NVFLARE_POC_WORKSPACE` to modify the workspace location. 

As mentioned earlier, the `prepare` subcommand calls internally the provisioning command. It is optional to specify a configuration file: a default template will be used to provision startup kits for 1 server, 2 clients and 1 admin. But since we already have an example configuration file in [`files/project.yml`/](files/project.yml/), let go ahead and use it:

In [ ]:
!rm -rf ./temp-workspace/
!export NVFLARE_POC_WORKSPACE=$(pwd -P)/"./temp-workspace/" && nvflare poc prepare -i files/project.yml

Running this command will create a new `temp-workspace` folder and generate an `example_project` within which we can find the startup kits for 1 server, 2 clients and 1 admin user:

In [ ]:
!tree ./temp-workspace -L 4


Let's walk through together how to launch a federated application in PoC mode step-by-step. 

> **NOTE**:
> When running the PoC mode, it's necessary to use a separate terminal since the several of the subcommands will run in the foreground emitting output from the server and any connected clients.

### 1. Create a Workspace

The first step is to create a workspace for the PoC. We can use the `nvflare poc prepare` for this. 

### 2. Start the PoC Mode

Next, let's use `nvflare poc start` to start the PoC mode:

In [ ]:
!nvflare poc start -h

When starting the PoC mode, it's necessary to use a separate terminal since the `nvflare poc start` subcommand will run in the foreground emitting output from the server and any connected clients.

Also note that `nvflare poc start` starts all participants by default, including the admin.  It's often nice to start th admin separately from the server and clients, so that we can interact with the FL system as admin later. To do this, we'll pass the `-ex admin@nvidia.com` argument to exclude the admin from the this PoC run and connect as admin later.

So pop open the launcher, launch a terminal, and run the following command:

```shell
export NVFLARE_POC_WORKSPACE=$(pwd -P)/notebooks/poc_workspace
export NVFLARE_HOME=$(pwd -P)/NVFlare
nvflare poc start -ex admin@nvidia.com
```

Keep this terminal open so you can continue to watch server and client output.

In [ ]:
!nvflare poc start -ex admin@nvidia.com

In [ ]:
from nvflare.fuel.flare_api.flare_api import new_insecure_session

admin_session = new_insecure_session(startup_kit_location = "./temp-workspace/example_project/prod_00/admin@nvidia.com")
print(admin_session.get_system_info())

### Preparing the POC environment
Before running POC mode, there are a couple important environment variables that should be set.

First, to simplify deploying the example apps in the NVFlare GitHub repo, you can set `NVFLARE_HOME` to the root of the GitHub clone.  In this case, we've cloned to our current working directory, so we can set it as:

In [ ]:
import os
workdir=os.getcwd()
%env NVFLARE_HOME={workdir}/../NVFlare

By default, POC mode uses a temporary workspace in /tmp/nvflare/poc.  We would like to keep the workspace within our working directory, so let's create a poc_workspace dir.  We can then use the `NVFLARE_POC_WORKSPACE` variable to define this as the POC workspace.

Note:  if you have previously created the poc_workspace, you will want to clean it up using the `nvflare poc --clean` command or manually remove the `poc_workspace` directory.

In [ ]:
# !nvflare poc --clean
!rm -r poc_workspace
!mkdir poc_workspace
%env NVFLARE_POC_WORKSPACE={workdir}/poc_workspace

### Preparing the POC workspace

Now that we've configured out POC environment, we can prepare the POC workspace.  By default, this will generate POC packages for a server and two clients.

(Note that `nvflare poc --prepare` prompts you to create the workspace.)

In [ ]:
!printf '%s\n' y | nvflare poc --prepare

Let's take a look.

In [ ]:
!tree poc_workspace

### Running the POC Deployment

When starting the POC deployment, it's necessary to use a separate terminal since the `nvflare poc --start` command will run  in the foreground emitting output from the server and any connected clients.

Also note that `nvflare poc --start` starts all participants, including the admin console.  It's often nice to start server and clients separately so that we can interact with the deployment using a separate admin console.  To do this, we'll pass the `-ex admin` arg to exclude the admin client from the initial POC run and use the FLARE API to run admin commands separately.

So pop open the launcher, launch a terminal, and run (remembering to set the NVFLARE_POC_WORKSPACE and NVFLARE_HOME vars!):

```shell
export NVFLARE_POC_WORKSPACE=$(pwd -P)/notebooks/poc_workspace
export NVFLARE_HOME=$(pwd -P)/NVFlare
nvflare poc --start -ex admin
```

Keep this terminal open so you can continue to watch server and client output.

### Using the FLARE API to connect to the POC deployment

The admin directory contains the startup script for the FLARE Console, which can be used interactively to operate a running FLARE deployment.  A FLARE deployment can also be managed using the FLARE API, which will use the configuration in the admin directory to connect to the FLARE server.  Since we already have the server and clients running in the background from the above terminal commands, we'll use FLARE API to start a new admin session and connect.

To get started, we need to import the FLARE API class and initialize session.

In [ ]:
from nvflare.fuel.flare_api.flare_api import new_insecure_session

admin_session = new_insecure_session(startup_kit_location = workdir + "/poc_workspace/admin")
print(admin_session.get_system_info())

### Launching a job with the FLARE API
Next we can use the FLARE API to launch one of the hello-world examples and monitor its status.  Note the difference here as compared to the Simulator example.  Because we're running a POC deployment with unique workspaces for the FLARE server and clients, we don't need to define a local workspace for job results before submitting the job.

All that's required to launch the job is the path to the job configuration.  This configuration is pushed to the server workspace and deployed to clients, and all job results are collected back in the server workspace.  After the job completes, we can use the FLARE API to download the results of the job from the server workspace to our admin directory.


In [ ]:
!tree /flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt

Before starting the job, we can save a bit of time by staging the CIFAR10 dataset.  We can use the helper script in the CIFAR10 example to do this:

In [ ]:
!bash /flare/NVFlare/examples/advanced/cifar10/cifar10-real-world/prepare_data.sh
!mkdir /root/data
!cp -rf /tmp/cifar10/* /root/data/

In [ ]:
path_to_job_config = "/flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt"
job_id = admin_session.submit_job(path_to_job_config)
print("Submitted job with job ID" + job_id)

### Monitoring the state of the FLARE deployment and job status

Now that the job is submitted, we can use the FLARE API to query the state of the system, show job status, and display job metadata.  These capabilities are especially useful through the course of a FLARE experiment, when you typically execute multiple jobs through the course of the course of the experiment.

For example, you can query the state of all jobs (with optional detailed output including job metadata), or query the metadata for a specific job by ID.

In [ ]:
import json

# Job Status
jobs_output = admin_session.list_jobs()
jobs_detail = admin_session.list_jobs(detailed=True)
print("Job Status")
print(json.dumps((jobs_output), indent=2))
print("\nJob Detail")
print(json.dumps((jobs_detail), indent=2))

# Job Metadata
print("\nJob Metadata")
admin_session.get_job_meta(job_id)

### Monitoring a job run with a callback function
You can also construct a simple callback function to monitor job status during a run.

In [ ]:
from nvflare.fuel.flare_api.flare_api import Session

def sample_cb(
        session: Session, job_id: str, job_meta, *cb_args, **cb_kwargs
    ) -> bool:
    if job_meta["status"] == "RUNNING":
        if cb_kwargs["cb_run_counter"]["count"] < 3:
            print(job_meta)
            print(cb_kwargs["cb_run_counter"])
        else:
            print(".", end="")
    else:
        print("\n" + str(job_meta))
    
    cb_kwargs["cb_run_counter"]["count"] += 1
    return True

admin_session.monitor_job(job_id, cb=sample_cb, cb_run_counter={"count":0})

### Retrieving job results
When the cell above monitoring job output shows that the job has finished with `<MonitorReturnCode.JOB_FINISHED: 0>`, we can use the FLARE API to download job results.  This is useful when running the FLARE API on a remote deployment and you wish to review job artifacts on your local laptop or workstation.

In [ ]:
job_download = admin_session.download_job_result(job_id)
print("Job download path: " + job_download)

In [ ]:
!tree {job_download}

In [ ]:
!tail {job_download}/workspace/log.txt

In [ ]:
cross_val_file = open(job_download + "/workspace/cross_site_val/cross_val_results.json")
cross_val_json = json.load(cross_val_file)
print(json.dumps(cross_val_json, indent=2))

### Stopping the POC deployment (important!)
Once the job has completed, we can stop the server and clients in the POC deployent.  This is necessary to free up ports for the following notebook examples.

In [ ]:
!nvflare poc stop

# Exercise 1

Restart the CIFAR10 project in PoC mode. Then use FLARE API to establish an admin user connection. After that, submit the job, query job status and download the job result using FLARE API. For references, [here](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.fuel.flare_api.flare_api.html#module-nvflare.fuel.flare_api.flare_api) is the documentation for FLARE API.

## Solution to Exercise 1

# Exercise 2

## Solution to Exercise 2